In [1]:
import torch, math, time
import torch.nn.functional as F
from torch import nn, Tensor
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from wzh.transformer import Transformer

torch.backends.cudnn.benchmark = False

print(torch.__version__)
device = "cuda"

n_class = 10

patch_shape = (4, 4)
d_patch = math.prod(patch_shape) * 3
n_patch = 3 * 32 * 32 // d_patch

dropout = 0.0
epochs = 20
batch_size = 384
learn_rate = 1e-4

2.7.1+cu128


In [ ]:
class BaselineLarge(nn.Module):
    def __init__(self):
        super().__init__()
        dim_model = 768
        self.embedding = nn.Linear(d_patch, dim_model)
        self.model = Transformer(
            nlayer=8,
            dim_model=dim_model,
            num_head=16,
            max_seq_len=n_patch,
            glu_attn=False,
        )
        self.output = nn.Linear(dim_model, n_class)

    def forward(self, x):
        x = F.unfold(x, patch_shape, stride=patch_shape).mT
        x = self.embedding(x)
        x = self.model(x)
        x = self.output(x)
        x = x.mean(-1)
        return x


class GLUAttentionLarge(nn.Module):
    def __init__(self):
        super().__init__()
        dim_model = 768
        self.embedding = nn.Linear(d_patch, dim_model)
        self.model = Transformer(
            nlayer=8,
            dim_model=dim_model,
            num_head=16,
            max_seq_len=n_patch,
            glu_attn=True,
        )
        self.output = nn.Linear(dim_model, n_class)

    def forward(self, x):
        x = F.unfold(x, patch_shape, stride=patch_shape).mT
        x = self.embedding(x)
        x = self.model(x)
        x = self.output(x)
        x = x.mean(-1)
        return x

In [3]:
def train(dataloader, model, loss_fn, optimizer):
    total_time = time.time()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.train()
    avg_loss, acc = 0, 0
    for i, (source, target) in enumerate(dataloader):
        source: Tensor = source.to(device, non_blocking=True)
        target: Tensor = target.to(device, non_blocking=True)
        pred = model(source)
        loss = loss_fn(pred, target)
        loss.backward(), optimizer.step(), optimizer.zero_grad()
        with torch.no_grad():
            avg_loss += loss.item()
            acc += (pred.argmax(1) == target).type(torch.float).sum().item()
    avg_loss /= num_batches
    acc /= size
    total_time = time.time() - total_time
    return (acc, avg_loss, total_time)


def train_amp(dataloader, model, loss_fn, optimizer):
    total_time = time.time()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.train()
    avg_loss, acc = 0, 0
    scaler = torch.amp.GradScaler('cuda')
    for i, (source, target) in enumerate(dataloader):
        source: Tensor = source.to(device, non_blocking=True)
        target: Tensor = target.to(device, non_blocking=True)
        with torch.amp.autocast('cuda',dtype=torch.bfloat16):
            pred = model(source)
            loss = loss_fn(pred, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        with torch.no_grad():
            avg_loss += loss.item()
            acc += (pred.argmax(1) == target).type(torch.float).sum().item()
    avg_loss /= num_batches
    acc /= size
    total_time = time.time() - total_time
    return (acc, avg_loss, total_time)


def val(dataloader, model, loss_fn):
    total_time = time.time()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    avg_loss, acc = 0, 0
    with torch.no_grad():
        for source, target in dataloader:
            source: Tensor = source.to(device, non_blocking=True)
            target: Tensor = target.to(device, non_blocking=True)
            pred = model(source)
            avg_loss += loss_fn(pred, target).item()
            acc += (pred.argmax(1) == target).type(torch.float).sum().item()
    avg_loss /= num_batches
    acc /= size
    total_time = time.time() - total_time
    return (acc, avg_loss, total_time)


transform_train = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.RandAugment(num_ops=2, magnitude=10),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        transforms.RandomErasing(
            p=0.2, scale=(0.02, 0.33), ratio=(0.3, 3.3), value=0, inplace=False
        ),
    ]
)

transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)
train_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=transform_train,
)
val_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=transform_test,
)

In [5]:
model_list: list[nn.Module] = [BaselineLarge, GLUAttentionLarge]

for model_creator in model_list:
    for i in range(10):
        model = model_creator().to(device)
        if i == 0:
            print(model)

        train_dataloader = DataLoader(
            train_data, batch_size, shuffle=True, num_workers=4, pin_memory=True
        )
        val_dataloader = DataLoader(
            val_data, batch_size, num_workers=4, pin_memory=True
        )

        loss_fn = nn.CrossEntropyLoss()
        optimizer = torch.optim.AdamW(model.parameters(), learn_rate)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)

        train_total_time = 0
        val_total_time = 0
        print(
            "epoch, train acc,   val acc,train loss,  val loss,train time,  val time,total time"
        )
        for epoch in range(1, epochs + 1):
            (train_acc, train_loss, train_time) = train_amp(
                train_dataloader, model, loss_fn, optimizer
            )
            scheduler.step()
            (val_acc, val_loss, val_time) = val(val_dataloader, model, loss_fn)
            train_total_time += train_time
            val_total_time += val_time

            print(
                f"{epoch:>5},{train_acc:>10.3f},{val_acc:>10.3f},{train_loss:>10f},{val_loss:>10f},{train_time:>10.1f},{val_time:>10.1f},{train_total_time + val_total_time:>10.1f}"
            )

BaselineLarge(
  (embedding): Linear(in_features=48, out_features=768, bias=True)
  (model): Transformer(
    (pe): PositionalEncoding()
    (layers): ModuleList(
      (0-7): 8 x TransformerLayer(
        (attn_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): MultiHeadSelfAttention(
          (wq): Linear(in_features=768, out_features=768, bias=True)
          (wk): Linear(in_features=768, out_features=768, bias=True)
          (wv): Linear(in_features=768, out_features=768, bias=True)
          (wo): Linear(in_features=768, out_features=768, bias=True)
        )
        (ffn_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (ffn): GLUFeedForward(
          (linear1): Linear(in_features=768, out_features=4096, bias=True)
          (linear2): Linear(in_features=2048, out_features=768, bias=True)
        )
      )
    )
  )
  (output): Linear(in_features=768, out_features=10, bias=True)
)
epoch, train acc,   val acc,train loss,  val loss,tr

In [ ]:

epoch, train acc,   val acc,train loss,  val loss,train time,  val time,total time
    1,     0.172,     0.280,  2.203265,  1.914788,      37.4,      15.3,      52.8
    2,     0.292,     0.404,  1.873082,  1.604431,      37.4,      15.3,     105.5
    3,     0.397,     0.509,  1.642942,  1.375762,      37.4,      15.3,     158.3
    4,     0.465,     0.567,  1.483620,  1.210087,      37.4,      15.3,     211.0
    5,     0.523,     0.577,  1.327212,  1.174123,      37.5,      15.3,     263.8
    6,     0.555,     0.610,  1.236765,  1.108251,      37.5,      15.3,     316.6
    7,     0.588,     0.660,  1.156704,  0.937983,      37.5,      15.3,     369.4
    8,     0.615,     0.691,  1.081717,  0.860842,      37.4,      15.3,     422.2
    9,     0.636,     0.703,  1.020992,  0.820908,      37.5,      15.3,     474.9
   10,     0.653,     0.719,  0.977842,  0.792908,      37.5,      15.2,     527.6
   11,     0.669,     0.740,  0.935308,  0.736373,      37.0,      14.9,     579.5
   12,     0.683,     0.751,  0.891163,  0.703001,      37.4,      15.3,     632.2
   13,     0.694,     0.760,  0.867490,  0.677005,      37.5,      15.3,     685.0
   14,     0.706,     0.765,  0.827172,  0.658617,      37.4,      15.3,     737.8
   15,     0.718,     0.769,  0.796743,  0.641177,      37.5,      15.3,     790.6
   16,     0.724,     0.781,  0.776860,  0.618119,      37.5,      15.3,     843.3
   17,     0.731,     0.782,  0.755233,  0.615422,      37.4,      15.3,     896.1
   18,     0.737,     0.784,  0.744243,  0.609853,      37.5,      15.3,     948.9
   19,     0.740,     0.785,  0.733618,  0.605345,      37.5,      15.3,    1001.7
   20,     0.741,     0.787,  0.732227,  0.603530,      37.5,      15.3,    1054.4
epoch, train acc,   val acc,train loss,  val loss,train time,  val time,total time
    1,     0.173,     0.287,  2.200623,  1.883137,      37.4,      15.3,      52.8
    2,     0.291,     0.410,  1.867760,  1.597549,      37.4,      15.3,     105.5
    3,     0.392,     0.497,  1.648468,  1.397075,      37.4,      15.3,     158.3
    4,     0.469,     0.581,  1.471326,  1.169980,      37.5,      15.3,     211.1
    5,     0.524,     0.611,  1.326534,  1.086228,      37.4,      15.3,     263.9
    6,     0.566,     0.637,  1.216454,  1.022974,      37.4,      15.3,     316.6
    7,     0.593,     0.674,  1.141082,  0.916477,      37.4,      15.3,     369.4
    8,     0.622,     0.691,  1.066730,  0.868388,      37.5,      15.3,     422.1
    9,     0.636,     0.715,  1.024849,  0.798639,      37.4,      15.3,     474.9
   10,     0.656,     0.725,  0.971409,  0.766290,      37.5,      15.3,     527.7
   11,     0.669,     0.737,  0.930992,  0.737741,      37.4,      15.3,     580.5
   12,     0.686,     0.747,  0.893063,  0.711150,      37.4,      15.3,     633.2
   13,     0.697,     0.749,  0.859311,  0.690948,      37.4,      15.3,     686.0
   14,     0.711,     0.761,  0.821945,  0.663436,      37.4,      15.3,     738.8
   15,     0.717,     0.766,  0.793528,  0.641470,      37.0,      15.0,     790.8
   16,     0.724,     0.775,  0.772736,  0.629723,      35.6,      14.0,     840.3
   17,     0.731,     0.774,  0.755637,  0.628365,      36.1,      14.3,     890.7
   18,     0.736,     0.777,  0.739012,  0.618674,      38.7,      15.9,     945.3
   19,     0.740,     0.780,  0.732723,  0.613314,      37.7,      15.0,     998.0
   20,     0.740,     0.780,  0.727099,  0.610260,      37.7,      15.3,    1051.0
epoch, train acc,   val acc,train loss,  val loss,train time,  val time,total time
    1,     0.173,     0.275,  2.199769,  1.872122,      39.0,      15.9,      55.0
    2,     0.295,     0.417,  1.864788,  1.598145,      38.3,      15.2,     108.5
    3,     0.386,     0.479,  1.653993,  1.421402,      38.2,      15.6,     162.2
    4,     0.457,     0.567,  1.496386,  1.210484,      37.8,      15.0,     214.9
    5,     0.513,     0.606,  1.352785,  1.108903,      37.7,      15.0,     267.7
    6,     0.556,     0.644,  1.237253,  0.998253,      37.7,      15.0,     320.4
    7,     0.585,     0.672,  1.161795,  0.929540,      37.7,      15.7,     373.8
    8,     0.616,     0.692,  1.084083,  0.862309,      38.7,      15.0,     427.5
    9,     0.632,     0.697,  1.039136,  0.837919,      37.7,      15.0,     480.2
   10,     0.653,     0.721,  0.981081,  0.787093,      37.9,      14.8,     532.9
   11,     0.667,     0.731,  0.940158,  0.763862,      37.2,      15.0,     585.1
   12,     0.680,     0.743,  0.902408,  0.730412,      37.7,      14.9,     637.6
   13,     0.693,     0.746,  0.867433,  0.706419,      37.5,      15.0,     690.2
   14,     0.703,     0.761,  0.837070,  0.672829,      37.0,      14.7,     741.9
   15,     0.714,     0.768,  0.810262,  0.654071,      37.0,      14.7,     793.6
   16,     0.723,     0.775,  0.782308,  0.641006,      37.2,      14.8,     845.5
   17,     0.728,     0.779,  0.767130,  0.627215,      37.1,      15.3,     898.0
   18,     0.733,     0.780,  0.753779,  0.622009,      37.4,      15.3,     950.8
   19,     0.737,     0.782,  0.741972,  0.616896,      37.5,      15.3,    1003.5
   20,     0.738,     0.783,  0.736942,  0.614779,      37.5,      15.3,    1056.3
epoch, train acc,   val acc,train loss,  val loss,train time,  val time,total time
    1,     0.170,     0.277,  2.204150,  1.876782,      37.5,      15.3,      52.8
    2,     0.299,     0.417,  1.859260,  1.578517,      37.4,      15.3,     105.6
    3,     0.395,     0.488,  1.646325,  1.404311,      37.5,      15.3,     158.3
    4,     0.467,     0.524,  1.479917,  1.339042,      37.4,      15.3,     211.1
    5,     0.513,     0.617,  1.350614,  1.084319,      37.5,      15.3,     263.9
    6,     0.556,     0.644,  1.237124,  0.978689,      37.5,      15.3,     316.6
    7,     0.587,     0.671,  1.154735,  0.925540,      37.5,      15.3,     369.4
    8,     0.611,     0.691,  1.094561,  0.865692,      37.4,      15.3,     422.1
    9,     0.631,     0.700,  1.036816,  0.834785,      37.5,      15.3,     474.9
   10,     0.647,     0.726,  0.988139,  0.770793,      37.5,      15.3,     527.7
   11,     0.665,     0.742,  0.944731,  0.726106,      37.5,      15.3,     580.5
   12,     0.682,     0.750,  0.902019,  0.704399,      37.5,      15.3,     633.3
   13,     0.691,     0.758,  0.868869,  0.687646,      37.4,      15.3,     686.0
   14,     0.705,     0.771,  0.829767,  0.659333,      37.5,      15.3,     738.8
   15,     0.715,     0.775,  0.811206,  0.640118,      37.4,      15.3,     791.5
   16,     0.720,     0.778,  0.790563,  0.635833,      37.5,      15.3,     844.3
   17,     0.729,     0.782,  0.761493,  0.621743,      37.5,      15.3,     897.1
   18,     0.737,     0.788,  0.748084,  0.609756,      37.5,      15.3,     949.9
   19,     0.737,     0.788,  0.741893,  0.604066,      37.5,      15.3,    1002.7
   20,     0.736,     0.788,  0.737721,  0.604132,      37.5,      15.3,    1055.5
epoch, train acc,   val acc,train loss,  val loss,train time,  val time,total time
    1,     0.176,     0.251,  2.189994,  1.932074,      37.5,      15.3,      52.8
    2,     0.296,     0.414,  1.858792,  1.578803,      37.5,      15.3,     105.5
    3,     0.392,     0.474,  1.644879,  1.454773,      37.5,      15.3,     158.3
    4,     0.467,     0.548,  1.481527,  1.279097,      37.5,      15.3,     211.1
    5,     0.519,     0.608,  1.339164,  1.093248,      37.5,      15.3,     263.9
    6,     0.560,     0.638,  1.227301,  1.009734,      37.5,      15.3,     316.7
    7,     0.589,     0.666,  1.150772,  0.939240,      37.5,      15.3,     369.4
    8,     0.612,     0.682,  1.089041,  0.885629,      37.5,      15.3,     422.2
    9,     0.636,     0.696,  1.025637,  0.842165,      37.4,      15.3,     475.0
   10,     0.648,     0.723,  0.987292,  0.786759,      37.5,      15.3,     527.7
   11,     0.666,     0.728,  0.940130,  0.758481,      37.5,      15.3,     580.5
   12,     0.683,     0.738,  0.892010,  0.738535,      37.5,      15.3,     633.3
   13,     0.694,     0.754,  0.863715,  0.690452,      37.5,      15.3,     686.1
   14,     0.705,     0.761,  0.833942,  0.667252,      37.5,      15.3,     738.9
   15,     0.715,     0.771,  0.800667,  0.649501,      37.4,      15.3,     791.6
   16,     0.723,     0.776,  0.785291,  0.634561,      37.5,      15.3,     844.4
   17,     0.730,     0.783,  0.761944,  0.622269,      37.5,      15.3,     897.1
   18,     0.734,     0.780,  0.750790,  0.616842,      37.5,      15.3,     949.9
   19,     0.739,     0.781,  0.742043,  0.611771,      37.5,      15.3,    1002.7
   20,     0.737,     0.784,  0.740447,  0.609510,      37.5,      15.3,    1055.5
epoch, train acc,   val acc,train loss,  val loss,train time,  val time,total time
    1,     0.174,     0.256,  2.198592,  1.949127,      37.5,      15.3,      52.8
    2,     0.294,     0.387,  1.873926,  1.677565,      36.8,      14.2,     103.8
    3,     0.380,     0.493,  1.679141,  1.394456,      35.8,      14.8,     154.4
    4,     0.451,     0.561,  1.515477,  1.245265,      36.8,      14.3,     205.6
    5,     0.515,     0.600,  1.355789,  1.119015,      36.1,      14.4,     256.1
    6,     0.553,     0.645,  1.245744,  1.003167,      36.4,      14.6,     307.1
    7,     0.584,     0.667,  1.166537,  0.935635,      36.5,      14.3,     358.0
    8,     0.606,     0.674,  1.102478,  0.892486,      36.5,      14.5,     409.0
    9,     0.630,     0.707,  1.041431,  0.820995,      36.6,      14.2,     459.8
   10,     0.648,     0.721,  0.995897,  0.789902,      35.3,      14.5,     509.7
   11,     0.664,     0.731,  0.950210,  0.757104,      36.5,      14.3,     560.6
   12,     0.674,     0.741,  0.913687,  0.719140,      37.1,      14.5,     612.2
   13,     0.689,     0.745,  0.873715,  0.707130,      36.8,      14.3,     663.3
   14,     0.703,     0.758,  0.842128,  0.675863,      35.1,      14.8,     713.2
   15,     0.709,     0.765,  0.820904,  0.655513,      35.7,      14.0,     762.9
   16,     0.717,     0.770,  0.798441,  0.638839,      35.2,      13.9,     812.0
   17,     0.726,     0.777,  0.774618,  0.625819,      35.1,      13.9,     861.1
   18,     0.732,     0.781,  0.761324,  0.622870,      35.1,      14.0,     910.1
   19,     0.731,     0.781,  0.759422,  0.615624,      35.1,      14.0,     959.2
   20,     0.733,     0.783,  0.756772,  0.613704,      35.9,      14.3,    1009.4
epoch, train acc,   val acc,train loss,  val loss,train time,  val time,total time
    1,     0.170,     0.259,  2.213849,  1.933299,      36.3,      13.9,      50.2
    2,     0.288,     0.404,  1.876821,  1.600945,      35.9,      14.0,     100.2
    3,     0.381,     0.491,  1.667752,  1.396396,      35.4,      14.2,     149.8
    4,     0.458,     0.573,  1.494560,  1.197309,      35.4,      14.8,     199.9
    5,     0.511,     0.595,  1.353749,  1.148938,      36.9,      15.0,     251.8
    6,     0.560,     0.637,  1.232332,  1.013397,      36.1,      14.2,     302.1
    7,     0.588,     0.664,  1.154600,  0.917998,      36.5,      14.1,     352.7
    8,     0.614,     0.686,  1.084523,  0.874937,      35.6,      14.7,     403.0
    9,     0.634,     0.715,  1.024365,  0.816365,      37.3,      14.5,     454.8
   10,     0.653,     0.710,  0.972854,  0.807522,      36.3,      14.3,     505.4
   11,     0.668,     0.735,  0.934306,  0.742692,      36.0,      14.3,     555.7
   12,     0.683,     0.745,  0.890591,  0.713060,      36.0,      14.5,     606.3
   13,     0.696,     0.754,  0.859415,  0.695440,      35.1,      13.9,     655.3
   14,     0.708,     0.769,  0.823889,  0.655137,      35.1,      13.9,     704.3
   15,     0.717,     0.773,  0.796531,  0.637849,      35.1,      13.9,     753.4
   16,     0.726,     0.773,  0.772833,  0.633635,      35.1,      14.0,     802.4
   17,     0.732,     0.781,  0.759733,  0.617147,      35.1,      13.9,     851.4
   18,     0.738,     0.787,  0.742126,  0.607887,      35.1,      13.9,     900.4
   19,     0.737,     0.786,  0.734702,  0.605684,      35.1,      13.9,     949.4
   20,     0.740,     0.787,  0.733342,  0.601684,      35.1,      13.9,     998.4
epoch, train acc,   val acc,train loss,  val loss,train time,  val time,total time
    1,     0.173,     0.257,  2.208649,  1.895155,      35.1,      13.9,      49.0
    2,     0.295,     0.411,  1.868327,  1.613142,      35.1,      13.9,      98.0
    3,     0.385,     0.479,  1.665347,  1.442349,      35.1,      13.9,     147.0
    4,     0.450,     0.538,  1.508760,  1.271873,      35.1,      13.9,     196.1
    5,     0.500,     0.589,  1.388525,  1.143359,      35.1,      13.9,     245.0
    6,     0.545,     0.616,  1.273135,  1.072184,      35.1,      14.4,     294.5
    7,     0.573,     0.652,  1.186492,  0.962016,      35.4,      13.9,     343.8
    8,     0.602,     0.686,  1.116955,  0.893720,      35.1,      13.9,     392.9
    9,     0.625,     0.692,  1.053713,  0.880377,      35.1,      13.9,     441.9
   10,     0.639,     0.711,  1.013086,  0.823842,      35.1,      13.9,     490.9
   11,     0.658,     0.715,  0.966419,  0.786556,      35.1,      13.9,     539.9
   12,     0.671,     0.737,  0.925732,  0.734937,      37.3,      15.3,     592.5
   13,     0.688,     0.752,  0.879134,  0.703593,      37.5,      15.3,     645.3
   14,     0.695,     0.751,  0.861980,  0.687448,      37.5,      15.3,     698.1
   15,     0.710,     0.764,  0.818443,  0.674329,      37.5,      15.3,     750.9
   16,     0.715,     0.771,  0.802945,  0.658557,      37.5,      15.3,     803.6
   17,     0.719,     0.775,  0.794039,  0.639740,      37.5,      15.3,     856.4
   18,     0.728,     0.778,  0.769070,  0.634998,      37.4,      15.3,     909.2
   19,     0.731,     0.774,  0.757337,  0.634082,      37.5,      15.3,     962.0
   20,     0.733,     0.776,  0.757243,  0.632252,      37.5,      15.3,    1014.8
epoch, train acc,   val acc,train loss,  val loss,train time,  val time,total time
    1,     0.172,     0.289,  2.211826,  1.878144,      37.5,      15.3,      52.8
    2,     0.290,     0.390,  1.880000,  1.648072,      37.5,      15.3,     105.6
    3,     0.383,     0.494,  1.662935,  1.389571,      37.4,      15.3,     158.3
    4,     0.455,     0.543,  1.505821,  1.264378,      37.5,      15.3,     211.1
    5,     0.511,     0.606,  1.358743,  1.107286,      37.5,      15.3,     264.0
    6,     0.555,     0.648,  1.245309,  0.985263,      37.4,      15.3,     316.7
    7,     0.582,     0.669,  1.169597,  0.913414,      37.5,      15.3,     369.5
    8,     0.610,     0.696,  1.095742,  0.863804,      37.5,      15.3,     422.3
    9,     0.628,     0.706,  1.046392,  0.827520,      37.5,      15.3,     475.1
   10,     0.649,     0.723,  0.989807,  0.777429,      37.5,      15.3,     527.8
   11,     0.665,     0.732,  0.950832,  0.759831,      37.4,      15.3,     580.6
   12,     0.679,     0.746,  0.909732,  0.718859,      37.5,      15.3,     633.4
   13,     0.690,     0.758,  0.869913,  0.693770,      37.5,      15.3,     686.2
   14,     0.705,     0.767,  0.837000,  0.665240,      37.5,      15.3,     738.9
   15,     0.714,     0.769,  0.804718,  0.653862,      37.5,      15.3,     791.7
   16,     0.725,     0.769,  0.781249,  0.645208,      37.5,      15.3,     844.5
   17,     0.730,     0.775,  0.762845,  0.636327,      37.5,      15.3,     897.3
   18,     0.736,     0.780,  0.746677,  0.623354,      37.4,      15.3,     950.0
   19,     0.736,     0.781,  0.746424,  0.617197,      37.4,      15.3,    1002.8
   20,     0.739,     0.783,  0.735987,  0.617178,      37.4,      15.3,    1055.5
epoch, train acc,   val acc,train loss,  val loss,train time,  val time,total time
    1,     0.172,     0.282,  2.208765,  1.872552,      37.4,      15.3,      52.7
    2,     0.293,     0.421,  1.860558,  1.572125,      37.5,      15.3,     105.5
    3,     0.389,     0.491,  1.652250,  1.394978,      37.4,      15.3,     158.3
    4,     0.463,     0.562,  1.486408,  1.240409,      37.4,      15.3,     211.0
    5,     0.516,     0.612,  1.353683,  1.080877,      37.5,      15.3,     263.8
    6,     0.553,     0.631,  1.249360,  1.027357,      37.5,      15.3,     316.6
    7,     0.584,     0.652,  1.165806,  0.943791,      37.5,      15.3,     369.4
    8,     0.611,     0.682,  1.100771,  0.887208,      37.5,      15.3,     422.2
    9,     0.633,     0.700,  1.039498,  0.831186,      37.5,      15.3,     474.9
   10,     0.645,     0.710,  0.995618,  0.805051,      37.4,      15.3,     527.7
   11,     0.665,     0.732,  0.943607,  0.755535,      37.5,      15.3,     580.5
   12,     0.679,     0.740,  0.906981,  0.735943,      37.5,      15.3,     633.2
   13,     0.689,     0.755,  0.874644,  0.687365,      37.4,      15.3,     686.0
   14,     0.704,     0.759,  0.838239,  0.677944,      37.5,      15.3,     738.8
   15,     0.712,     0.770,  0.812633,  0.645350,      37.5,      15.3,     791.6
   16,     0.720,     0.770,  0.792215,  0.650885,      37.5,      15.3,     844.3
   17,     0.727,     0.773,  0.771910,  0.641001,      37.5,      15.3,     897.1
   18,     0.731,     0.778,  0.760475,  0.625548,      37.5,      15.3,     949.9
   19,     0.736,     0.783,  0.743838,  0.617371,      37.5,      15.3,    1002.6
   20,     0.737,     0.781,  0.742632,  0.617788,      37.5,      15.3,    1055.4